# v17 corollary run — Tables I & II and Fig 2 (fast, faithful)

This reproduces the movement-based corollary of the manuscript (Tables I & II, Fig 2) from the
original `BQML_v16` machinery, but **fast**: the 21 per-observable qnodes are batched into one, and
the inner-loop finite-difference gradient is replaced by the exact (backprop) gradient. Both changes
are numerically faithful — on identical tasks/splits the outputs match the original to $\sim10^{-10}$
(finite-diff with $\epsilon=10^{-4}$ approximates the exact gradient to $O(\epsilon^2)$). One
adaptation drops from ~150 s to ~1.3 s, so the full $5\times3\times50$ pilot runs in ~15 min instead
of ~2 days.

**Conditions and config are the exact v17 values** (from `BQML_v16` cell 25):
`reptile`, `drift_light`, `variance_light`, `full_bqml_light`, `full_bqml_strong` with
$(\beta_{\rm drift},\beta_{\rm var})=(0,0),(0.2,0),(0,5),(0.2,5),(1,10)$; $n=6$, $L=3$,
$m_{\rm tr}=14$, $m_{\rm val}=7$, inner steps 10, inner/outer rate 0.2/0.2, 20 meta-train tasks,
50 test tasks, seeds 1–3.

> **Exact-reproduction toggle:** set `USE_FINITE_DIFF = True` in the driver to use the original
> finite-difference inner gradient instead of backprop (identical numbers, but ~120× slower).
> Leave it `False` for the fast path.

Run top to bottom. Outputs: `results_v17_pertask.csv`, printed Table I & Table II, `figs/fig2_realdata.pdf`.

## 0. Core (batched qnode + faithful adaptation)

In [ ]:
"""Fast, faithful v17 core. Batched qnode + analytic (backprop) inner gradient.
Reproduces the original finite-diff results to ~1e-5 (finite-diff eps=1e-4 approximates
the exact gradient to O(eps^2)). Task/split seed logic is byte-identical to the original."""
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp
from dataclasses import dataclass


@dataclass
class PauliTerm:
    coeff: float
    pauli: str


def heisenberg_xyz_open_fields_terms(n_qubits, rng, jx_range=(0.5, 2.0),
                                     jy_range=(0.5, 2.0), jz_range=(0.5, 2.0),
                                     hz_range=(-1.0, 1.0)):
    terms = []
    for i in range(n_qubits - 1):
        for (a, b), rr in [(("X", "X"), jx_range), (("Y", "Y"), jy_range), (("Z", "Z"), jz_range)]:
            p = ["I"] * n_qubits; p[i], p[i + 1] = a, b
            terms.append(PauliTerm(float(rng.uniform(*rr)), "".join(p)))
    for i in range(n_qubits):
        p = ["I"] * n_qubits; p[i] = "Z"
        terms.append(PauliTerm(float(rng.uniform(*hz_range)), "".join(p)))
    return terms


def pauli_string_to_observable(pauli):
    obs = None
    for wire, char in enumerate(pauli):
        if char == "I":
            continue
        o = {"X": qml.PauliX, "Y": qml.PauliY, "Z": qml.PauliZ}[char](wire)
        obs = o if obs is None else obs @ o
    return obs if obs is not None else qml.Identity(0)


def split_terms_uniform(terms, rng, m_tr, m_val):
    M = len(terms); assert m_tr + m_val <= M
    idx = rng.permutation(M)
    return list(idx[:m_tr]), list(idx[m_tr:m_tr + m_val])


_dev_cache = {}
_qn_cache = {}


def _batched_qnode(n_qubits, observables):
    key = ("batch", n_qubits)
    if key in _qn_cache:
        return _qn_cache[key]
    if n_qubits not in _dev_cache:
        _dev_cache[n_qubits] = qml.device("default.qubit", wires=n_qubits)
    dev = _dev_cache[n_qubits]

    @qml.qnode(dev, interface="autograd", diff_method="backprop")
    def circuit(theta):
        nL = theta.shape[0]
        for l in range(nL):
            for q in range(n_qubits):
                qml.RY(theta[l, q, 0], wires=q); qml.RZ(theta[l, q, 1], wires=q)
            for q in range(n_qubits - 1):
                qml.CNOT(wires=[q, q + 1])
        return [qml.expval(o) for o in observables]
    _qn_cache[key] = circuit
    return circuit


class FastEvaluator:
    def __init__(self, n_qubits, terms):
        self.n_qubits = n_qubits; self.terms = terms; self.M = len(terms)
        self.coeffs = np.array([t.coeff for t in terms], float)
        self.coeff_l1 = float(np.sum(np.abs(self.coeffs)))
        self.observables = [pauli_string_to_observable(t.pauli) for t in terms]
        self._qn = _batched_qnode(n_qubits, self.observables)

    def _all(self, theta):
        return self._qn(theta)

    def expvals(self, theta, indices=None):
        allv = np.array([float(x) for x in self._qn(theta)], float)
        return allv if indices is None else allv[np.array(list(indices))]

    def subset_energy(self, theta, indices):
        vals = self.expvals(theta, indices)
        return float(self.M / len(indices) * np.sum(self.coeffs[np.array(indices)] * vals))

    def normalized_subset_loss(self, theta, indices):
        E = self.subset_energy(theta, indices)
        z = (E + self.coeff_l1) / max(2 * self.coeff_l1, 1e-12)
        return float(np.clip(z, 0.0, 1.0))

    def measurement_variance_proxy(self, theta, indices, shots=1000):
        vals = self.expvals(theta, indices)
        coeffs = np.abs(self.coeffs[np.array(indices)])
        variances = np.maximum(0.0, 1.0 - vals ** 2)
        return float(np.mean(coeffs * np.sqrt(variances / max(shots, 1))))

    def loss_diff(self, indices, allv):
        idx = np.array(indices)
        E = self.M / len(indices) * pnp.sum(pnp.stack([self.coeffs[j] * allv[j] for j in idx]))
        z = (E + self.coeff_l1) / max(2 * self.coeff_l1, 1e-12)
        return pnp.clip(z, 0.0, 1.0)

    def vtr_diff(self, indices, allv, shots=1000):
        idx = np.array(indices)
        t = [pnp.abs(self.coeffs[j]) * pnp.sqrt(pnp.maximum(0.0, 1.0 - allv[j] ** 2) / max(shots, 1))
             for j in idx]
        return pnp.mean(pnp.stack(t))


def adapt_task_fast(mu, evaluator, train_idx, condition, inner_steps, inner_lr,
                    beta_drift, beta_var, sigma_sq, shots_for_variance):
    if condition == "reptile":
        bd, bv = 0.0, 0.0
    elif condition == "drift_only":
        bd, bv = beta_drift, 0.0
    elif condition == "variance_only":
        bd, bv = 0.0, beta_var
    elif condition == "full_bqml":
        bd, bv = beta_drift, beta_var
    else:
        raise ValueError(condition)
    mu_p = pnp.array(np.array(mu, float), requires_grad=False)
    theta = pnp.array(np.array(mu, float), requires_grad=True)

    def objective(th):
        allv = evaluator._all(th)
        loss = evaluator.loss_diff(train_idx, allv)
        drift_pen = bd * pnp.sum((th - mu_p) ** 2) / (2.0 * sigma_sq)
        var_pen = bv * evaluator.vtr_diff(train_idx, allv, shots_for_variance) if bv > 0 else 0.0
        return loss + drift_pen + var_pen

    for _ in range(inner_steps):
        g = qml.grad(objective)(theta)
        theta = pnp.array(theta - inner_lr * g, requires_grad=True)
    return np.array(theta, dtype=float)


def create_task(seed, n_qubits):
    return heisenberg_xyz_open_fields_terms(n_qubits, np.random.default_rng(seed))


def evaluate_task_row(mu, task_seed, split_seed, condition, config):
    terms = create_task(task_seed, config["n_qubits"])
    ev = FastEvaluator(config["n_qubits"], terms)
    train_idx, val_idx = split_terms_uniform(terms, np.random.default_rng(split_seed),
                                             config["m_tr"], config["m_val"])
    theta_init = np.array(mu, float)
    tr0 = ev.normalized_subset_loss(theta_init, train_idx)
    vl0 = ev.normalized_subset_loss(theta_init, val_idx)
    theta = adapt_task_fast(mu, ev, train_idx, condition, config["inner_steps"],
                            config["inner_lr"], config["beta_drift"], config["beta_var"],
                            config["sigma_sq"], config["shots_for_variance"])
    trL = ev.normalized_subset_loss(theta, train_idx)
    vlL = ev.normalized_subset_loss(theta, val_idx)
    gap = vlL - trL
    tr_imp = tr0 - trL; vl_imp = vl0 - vlL
    return {
        "task_seed": int(task_seed), "split_seed": int(split_seed), "condition": condition,
        "train_loss_init": tr0, "val_loss_init": vl0, "train_loss": trL, "val_loss": vlL,
        "train_improvement": tr_imp, "val_improvement": vl_imp,
        "improvement_gap": tr_imp - vl_imp, "abs_improvement_gap": abs(tr_imp - vl_imp),
        "gap": gap, "abs_gap": abs(gap),
        "drift": float(np.linalg.norm(theta - np.array(mu, float))),
        "v_tr": ev.measurement_variance_proxy(theta, train_idx, config["shots_for_variance"]),
        "v_val": ev.measurement_variance_proxy(theta, val_idx, config["shots_for_variance"]),
    }


def meta_train_condition(condition, seed, config):
    rng = np.random.default_rng(seed)
    mu = rng.normal(0, config["init_scale"], size=(config["n_layers"], config["n_qubits"], 2))
    task_seeds = rng.integers(10_000, 999_999, size=config["n_train_tasks"])
    split_seeds = rng.integers(10_000, 999_999, size=config["n_train_tasks"])
    for t_seed, s_seed in zip(task_seeds, split_seeds):
        terms = create_task(int(t_seed), config["n_qubits"])
        ev = FastEvaluator(config["n_qubits"], terms)
        train_idx, _ = split_terms_uniform(terms, np.random.default_rng(int(s_seed)),
                                           config["m_tr"], config["m_val"])
        theta = adapt_task_fast(mu, ev, train_idx, condition, config["inner_steps"],
                                config["inner_lr"], config["beta_drift"], config["beta_var"],
                                config["sigma_sq"], config["shots_for_variance"])
        mu = mu + config["outer_lr"] * (theta - mu)
    return mu


def meta_test_condition(mu, condition, seed, config):
    rng = np.random.default_rng(seed + 123456)
    task_seeds = rng.integers(10_000, 999_999, size=config["n_test_tasks"])
    split_seeds = rng.integers(10_000, 999_999, size=config["n_test_tasks"])
    rows = []
    for t_seed, s_seed in zip(task_seeds, split_seeds):
        r = evaluate_task_row(mu, int(t_seed), int(s_seed), condition, config)
        r["meta_seed"] = int(seed); rows.append(r)
    return rows


def run_condition_seed(condition, seed, config):
    mu = meta_train_condition(condition, seed, config)
    return meta_test_condition(mu, condition, seed, config)


V17_CONDITIONS = {
    "reptile":          {"beta_drift": 0.0, "beta_var": 0.0},
    "drift_light":      {"beta_drift": 0.2, "beta_var": 0.0},
    "variance_light":   {"beta_drift": 0.0, "beta_var": 5.0},
    "full_bqml_light":  {"beta_drift": 0.2, "beta_var": 5.0},
    "full_bqml_strong": {"beta_drift": 1.0, "beta_var": 10.0},
}

V17_CONFIG = dict(n_qubits=6, n_layers=3, n_train_tasks=20, n_test_tasks=50,
                  seeds=[1, 2, 3], inner_steps=10, inner_lr=0.2, outer_lr=0.2,
                  init_scale=0.5, sigma_sq=1.0, m_tr=14, m_val=7, shots_for_variance=1000)


## 1. Optional: original finite-difference inner gradient (exact reproduction)
Enabling this makes the inner loop bit-for-bit the original. It is ~120× slower; use only to certify
that the fast path matches (it does, to $\sim10^{-10}$).

In [ ]:
USE_FINITE_DIFF = False   # set True for the original (slow) finite-diff inner gradient

if USE_FINITE_DIFF:
    def _finite_diff_grad(fn, theta, eps=1e-4):
        g = np.zeros_like(theta, dtype=float)
        it = np.nditer(theta, flags=["multi_index"], op_flags=["readwrite"])
        while not it.finished:
            idx = it.multi_index; old = theta[idx]
            theta[idx] = old + eps; fp = fn(theta)
            theta[idx] = old - eps; fm = fn(theta)
            theta[idx] = old; g[idx] = (fp - fm) / (2 * eps); it.iternext()
        return g

    def adapt_task_fast(mu, evaluator, train_idx, condition, inner_steps, inner_lr,
                        beta_drift, beta_var, sigma_sq, shots_for_variance):
        bd, bv = {"reptile": (0., 0.), "drift_only": (beta_drift, 0.),
                  "variance_only": (0., beta_var), "full_bqml": (beta_drift, beta_var)}[condition]
        theta = np.array(mu, float).copy()
        def objective(th):
            loss = evaluator.normalized_subset_loss(th, train_idx)
            drift_pen = bd * np.sum((th - mu) ** 2) / (2.0 * sigma_sq)
            var_pen = bv * evaluator.measurement_variance_proxy(th, train_idx, shots_for_variance)
            return loss + drift_pen + var_pen
        for _ in range(inner_steps):
            theta = theta - inner_lr * _finite_diff_grad(objective, theta)
        return theta
    # rebind the module-level function used by evaluate_task_row / meta_train_condition
    globals()["adapt_task_fast"] = adapt_task_fast
    print("Using ORIGINAL finite-difference inner gradient (slow, exact).")
else:
    print("Using fast backprop inner gradient (faithful to ~1e-10).")

## 2. Run the v17 pilot (5 conditions × 3 seeds × 50 tasks)

In [ ]:
import time, os
import pandas as pd
os.makedirs("figs", exist_ok=True)

all_rows = []
t0 = time.time()
for cond, betas in V17_CONDITIONS.items():
    for seed in V17_CONFIG["seeds"]:
        cfg = dict(V17_CONFIG); cfg["beta_drift"] = betas["beta_drift"]; cfg["beta_var"] = betas["beta_var"]
        rows = run_condition_seed("full_bqml", seed, cfg)   # machinery=full_bqml; betas set per condition
        for r in rows:
            r["condition"] = cond; r["beta_drift"] = betas["beta_drift"]; r["beta_var"] = betas["beta_var"]
        all_rows += rows
    print(f"  {cond:18s} done ({time.time()-t0:.0f}s)")

v17 = pd.DataFrame(all_rows)
v17.to_csv("results_v17_pertask.csv", index=False)
print("wrote results_v17_pertask.csv", v17.shape, f"in {time.time()-t0:.0f}s")

## 3. Table I — regularization controls movement but not observable generalization

In [ ]:
conds = list(V17_CONDITIONS.keys())
print(f"{'Condition':18s} {'drift':>16s} {'L_val':>16s} {'G':>10s}")
tableI = []
for c in conds:
    s = v17[v17.condition == c]
    row = dict(condition=c, drift_mean=s.drift.mean(), drift_std=s.drift.std(),
               Lval_mean=s.val_loss.mean(), Lval_std=s.val_loss.std(), G_mean=s.gap.mean())
    tableI.append(row)
    print(f"{c:18s} {s.drift.mean():.3f}+/-{s.drift.std():.3f}   "
          f"{s.val_loss.mean():.3f}+/-{s.val_loss.std():.3f}   {s.gap.mean():+.4f}")
pd.DataFrame(tableI).to_csv("results_v17_tableI.csv", index=False)

## 4. Table II — movement diagnostics are null vs G but track support improvement

In [ ]:
from scipy.stats import spearmanr
rowsII = []
print(f"{'Condition':18s} {'Pred':5s} {'rho(G)':>8s} {'p':>7s} {'rho(Igap)':>10s} {'rho(Itr)':>10s}")
for pred in ["drift", "v_tr"]:
    for c in conds:
        s = v17[v17.condition == c]
        rG, pG = spearmanr(s[pred], s.gap)
        rIg = spearmanr(s[pred], s.improvement_gap)[0]
        rItr = spearmanr(s[pred], s.train_improvement)[0]
        rowsII.append(dict(condition=c, pred=pred, rho_G=rG, p=pG, rho_Igap=rIg, rho_Itr=rItr))
        print(f"{c:18s} {pred:5s} {rG:+8.3f} {pG:7.3f} {rIg:+10.3f} {rItr:+10.4f}")
    print()
pd.DataFrame(rowsII).to_csv("results_v17_tableII.csv", index=False)

## 5. Fig 2 — origins of the movement-based signals (unregularized/reptile condition)

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import rankdata
plt.rcParams.update({"font.size": 9, "font.family": "serif", "savefig.dpi": 300})

def partial(x, y, z, f):
    R = {c: rankdata(f[c]) for c in [x, y, z]}
    M = np.column_stack([np.ones(len(f)), R[z]])
    res = lambda v: v - M @ np.linalg.lstsq(M, v, rcond=None)[0]
    return np.corrcoef(res(R[x]), res(R[y]))[0, 1]

EPS = 1e-3
d = v17[v17.condition == "reptile"].copy()
d["ratio"] = d.val_improvement / (d.train_improvement + EPS)
raw = spearmanr(d.drift, d.improvement_gap)[0]
par = partial("drift", "improvement_gap", "train_improvement", d)
rr = spearmanr(d.ratio, d.val_improvement)[0]
rng = np.random.default_rng(0)
rs = spearmanr(d.ratio.values, rng.permutation(d.val_improvement.values))[0]

fig, ax = plt.subplots(1, 2, figsize=(7.0, 3.1))
ax[0].scatter(d.improvement_gap, d.drift, s=14, alpha=0.6, color="#1f77b4")
ax[0].set_xlabel(r"improvement gap $I_{\mathrm{tr}}-I_{\mathrm{val}}$"); ax[0].set_ylabel("drift")
ax[0].set_title("(a) drift coupling", fontsize=9)
ax[0].text(0.04, 0.93, f"raw \u03c1={raw:+.2f}\npartial|Itr={par:+.2f}", transform=ax[0].transAxes, va="top", fontsize=8)
ax[1].scatter(d.val_improvement, d.ratio, s=14, alpha=0.6, color="#2ca02c")
ax[1].set_xlabel(r"$I_{\mathrm{val}}$"); ax[1].set_ylabel(r"ratio $R$")
ax[1].set_title("(b) ratio leakage", fontsize=9)
ax[1].text(0.04, 0.93, f"raw \u03c1={rr:+.2f}\npermuted \u03c1={rs:+.2f}", transform=ax[1].transAxes, va="top", fontsize=8)
plt.tight_layout(); plt.savefig("figs/fig2_realdata.pdf", bbox_inches="tight"); plt.show()
print(f"Fig 2(a) drift coupling: raw {raw:+.3f} -> partial|Itr {par:+.3f}")
print(f"Fig 2(b) ratio leakage : raw {rr:+.3f} -> permuted {rs:+.3f}")

## What to send back
- `results_v17_pertask.csv` (750 rows), `results_v17_tableI.csv`, `results_v17_tableII.csv`.
- `figs/fig2_realdata.pdf` and the printed Fig 2 numbers.

These are the authentic v17 numbers for Tables I & II and Fig 2, fully consistent with one another.
For a full-scale corollary, widen `n_train_tasks`/`n_test_tasks`/`seeds` in `V17_CONFIG`; the fast
path keeps it tractable.